# Rate-KL SAE: firing-rate-targeted sparsity (post-Gini direction)

Design from the Gini post-mortem: steering impact needs untouched magnitudes (non-suppressive), and the hub pathology is a firing-RATE phenomenon that distribution-shape objectives (row/column Gini, load balancing) can't fix without collateral damage. `rate_kl_sae.py` penalizes each feature's firing *probability* toward a target rate rho via KL, computed from a smooth sigmoid surrogate on pre-ReLU activations -- magnitude never appears in the penalty. Expected active count per sample = N*rho (soft, differentiable, O(N) analogue of TopK's k). Hubs (p~1) and dead features (p~0) are both individually expensive.

**Win condition:** some lambda where sparsity lands near the TopK reference (~0.91), Top10Purity clearly beats TopK's ~0.036, and ablate/clamp beats TopK's ~0.064/~0.128 -- all three at once, which no Gini variant achieved.

**Failure modes to watch:** lambda too low -> penalty ignored, dense code (sparsity well below target); lambda too high -> firing rates pinned to rho but reconstruction starved (MSE spikes); tau mismatch -> p_hat estimates diverge from true firing rates (sparsity far from N*rho at matched lambda).

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# rho=0.09 targets ~92 active features per sample, matching the sparsity
# regime of all prior runs; the TopK reference (k=round(rho*1024)=92) is
# trained in the same run.
!python rate_kl_sae.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambdas 0.001 0.01 0.1 1.0 --tau 0.1

In [ ]:
import json
with open('results/rate_kl/fashion_mnist_seed0.json') as f:
    results = json.load(f)

print(f"{'Model':24s} {'MSE':>8s} {'Sparsity':>9s} {'ImpGini':>8s} {'Top10Share':>11s} "
      f"{'MeanPurity':>11s} {'Top10Purity':>12s} {'Ablate':>8s} {'Clamp':>8s}")
for name, r in results.items():
    print(f"{name:24s} {r['mse']:8.4f} {r['relative_sparsity']:9.3f} "
          f"{r['importance_gini_coefficient']:8.3f} {r['top10_importance_share']:11.3f} "
          f"{r['mean_purity']:11.3f} {r['mean_purity_top10pct_by_importance']:12.3f} "
          f"{r['steering_impact_ablate']:8.4f} {r['steering_impact_clamp']:8.4f}")
results

In [ ]:
!zip -r rate_kl_results.zip results/rate_kl
from google.colab import files
files.download('rate_kl_results.zip')